# 05 — Topology & Deep Dive

Examine operator centroids, territory maps, event horizons, and prototypical verbs.

**Requires:** Pre-computed data from the repository OR embeddings + classifications.

In [ ]:
# ═══ Load topology data ═══
import json
from pyodide.http import pyfetch

HELIX = ['NUL','DES','INS','SEG','CON','SYN','ALT','SUP','REC']

topology = None
operators_data = None

# Try loading pre-computed
for name, url in [('topology', '../output/topology_report.json'), ('operators', '../data/operators.json')]:
    try:
        resp = await pyfetch(url)
        data = json.loads(await resp.string())
        if name == 'topology': topology = data
        else: operators_data = data
        print(f'Loaded {name}')
    except:
        print(f'{name} not found')

ops = None
if topology and 'operators' in topology:
    ops = topology['operators']
    print(f'\nTopology data: {len([o for o in ops if o])} operators analyzed')
elif operators_data:
    ops = operators_data if isinstance(operators_data, list) else operators_data.get('operators', [])
    print(f'\nOperator data: {len(ops)} operators')

In [ ]:
# ═══ Operator territory overview ═══
if ops:
    print(f'{"Op":>5s} {"Total":>6s} {"Core":>6s} {"Settled":>8s} {"Contest":>8s} {"Foreign":>8s} {"Margin":>7s}')
    print('=' * 55)
    for op in ops:
        if op is None: continue
        t = op.get('territory', {})
        print(f'{op["operator"]:>5s} {op.get("total",0):>6d} '
              f'{t.get("core",0):>6d} {t.get("settled",0):>8d} '
              f'{t.get("contested",0):>8d} {t.get("foreign",0):>8d} '
              f'{op.get("margin_mean",0):>+7.4f}')
else:
    print('No topology data. Run the full pipeline or load pre-computed data.')

In [ ]:
# ═══ Prototypical verbs per operator ═══
if ops:
    for op in ops:
        if op is None: continue
        protos = op.get('prototypes', [])
        print(f'\n{op["operator"]} — {op.get("total",0)} verbs')
        print(f'  Prototypes (purest examples):')
        for p in protos[:5]:
            print(f'    {p["verb"]:20s} margin={p.get("margin",0):+.3f}')
            if p.get('reason'):
                print(f'      {p["reason"][:70]}')
        
        horizon = op.get('horizon', [])
        if horizon:
            print(f'  Event horizon (boundary verbs):')
            for h in horizon[:3]:
                print(f'    {h["verb"]:20s} <-> {h.get("neighbor","?")} margin={h.get("margin",0):+.3f}')

In [ ]:
# ═══ Nearest operator neighbors ═══
if ops:
    print('Operator proximity (centroid cosine similarity):')
    print()
    for op in ops:
        if op is None: continue
        neighbors = op.get('neighbors', [])
        if neighbors:
            nearest = neighbors[0]
            others = ', '.join(f'{n["op"]}={n["sim"]:.3f}' for n in neighbors[1:3])
            print(f'  {op["operator"]:>5s} nearest: {nearest["op"]} ({nearest["sim"]:.4f})  also: {others}')

In [ ]:
# ═══ Scale distribution per operator ═══
if ops:
    print('Scale distribution (what scope each operator acts at):')
    for op in ops:
        if op is None: continue
        scales = op.get('scales', {})
        if scales:
            scale_str = ', '.join(f'{k}:{v}' for k, v in list(scales.items())[:5])
            print(f'  {op["operator"]:>5s}: {scale_str}')